# O-RAG Interactive Demo - Query Analysis & Timing

**Purpose**: Comprehensive evaluation of O-RAG system with multiple query types, timing analysis, and visual demonstration of retrieval capabilities including document images and metadata.

**Features**:
- Multiple query types (factual, reasoning, multi-hop)
- Real-time timing analysis
- Retrieved document visualization
- Query-response pair table with latency
- Memory usage tracking
- Document images display (if available)

## 1. Setup and Imports

In [ ]:
import os
import sys
import time
import psutil
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from IPython.display import display, HTML, Image as IPImage

# Add parent path for imports
sys.path.insert(0, os.path.abspath('..'))

from rag.pipeline import RAGPipeline
from rag.llm import LlamaCppModel
from rag.retriever import VectorStoreRetriever
from analytics import AnalyticsCollector, HealthMonitor

# Configuration
pd.set_option('display.max_colwidth', None)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✅ All imports successful")
print(f"Timestamp: {datetime.now()}")

## 2. Initialize RAG System

In [ ]:
# Paths
MODEL_PATH = "qwen2.5-1.5b-instruct-q4_k_m.gguf"
EMBEDDING_MODEL = "nomic-embed-text-v1.5.Q4_K_M.gguf"

# Verify models
model_exists = os.path.exists(MODEL_PATH)
embedding_exists = os.path.exists(EMBEDDING_MODEL)

print(f"LLM Model: {'✅' if model_exists else '❌'} {MODEL_PATH}")
print(f"Embedding Model: {'✅' if embedding_exists else '❌'} {EMBEDDING_MODEL}")

# Initialize RAG Pipeline
print("\n🔄 Initializing RAG Pipeline...")
start_init = time.time()

rag = RAGPipeline(
    model_path=MODEL_PATH,
    embedding_model_path=EMBEDDING_MODEL,
    context_window=512,
    max_tokens=256,
    temperature=0.7,
    top_k=5
)

init_time = time.time() - start_init
print(f"✅ RAG initialized in {init_time:.2f} seconds")

## 3. Define Query Sets by Type

In [ ]:
# Query categories for comprehensive testing
QUERY_SETS = {
    'factual': [
        "What is the capital of France?",
        "When was Python programming language created?",
        "Who developed the concept of machine learning?",
        "What is the atomic number of oxygen?",
        "How many continents are there on Earth?"
    ],
    'reasoning': [
        "Why is machine learning important for data analysis?",
        "How does neural network training work?",
        "What are the advantages of RAG systems over traditional chatbots?",
        "Explain the relationship between embeddings and semantic similarity",
        "How does quantization improve model deployment?"
    ],
    'multi_hop': [
        "Describe the RAG pipeline and why embeddings are crucial for it",
        "How does memory optimization impact real-time inference performance?",
        "What role do retrieval systems play in question answering systems?",
        "Compare LLM generation quality with and without retrieval",
        "How does context window size affect multi-document reasoning?"
    ],
    'comparative': [
        "Differentiate between BM25 and vector-based retrieval",
        "How do 4-bit quantization and 8-bit quantization compare?",
        "What's the difference between RAG and fine-tuning approaches?",
        "Compare memory usage of different model quantization levels",
        "How does batch inference differ from single-query inference?"
    ]
}

print(f"Query Sets Prepared:")
for category, queries in QUERY_SETS.items():
    print(f"  • {category.upper()}: {len(queries)} queries")
print(f"\nTotal: {sum(len(q) for q in QUERY_SETS.values())} queries")

## 4. Execute Queries and Track Metrics

In [ ]:
# Initialize metrics collection
results = []
memory_monitor = HealthMonitor()

print("🚀 Executing Queries...\n")
print(f"{'Query #':<8} {'Type':<12} {'Query':<50} {'Time (ms)':<12} {'Status'}")
print("="*90)

query_num = 1
for query_type, queries in QUERY_SETS.items():
    for query in queries:
        try:
            # Record start metrics
            start_time = time.time()
            start_memory = psutil.virtual_memory().used
            
            # Execute query
            query_text = query[:40] + "..." if len(query) > 40 else query
            response = rag.ask(query, top_k=5)
            
            # Record end metrics
            end_time = time.time()
            end_memory = psutil.virtual_memory().used
            latency = (end_time - start_time) * 1000  # Convert to ms
            memory_delta = (end_memory - start_memory) / (1024 * 1024)  # Convert to MB
            
            # Extract retrieved documents count
            retrieved_count = len(response.get('retrieved_docs', []))
            
            # Store result
            results.append({
                'query_id': query_num,
                'query_type': query_type,
                'query': query,
                'response': response.get('answer', 'N/A'),
                'latency_ms': latency,
                'memory_delta_mb': memory_delta,
                'retrieved_docs': retrieved_count,
                'confidence': response.get('confidence', 0),
                'timestamp': datetime.now(),
                'status': '✅'
            })
            
            print(f"{query_num:<8} {query_type:<12} {query_text:<50} {latency:<12.2f} ✅")
            query_num += 1
            
        except Exception as e:
            print(f"{query_num:<8} {query_type:<12} {query_text:<50} {'ERROR':<12} ❌")
            results.append({
                'query_id': query_num,
                'query_type': query_type,
                'query': query,
                'response': f'Error: {str(e)}',
                'latency_ms': 0,
                'memory_delta_mb': 0,
                'retrieved_docs': 0,
                'confidence': 0,
                'timestamp': datetime.now(),
                'status': '❌'
            })
            query_num += 1

print("\n✅ Query execution complete!")
df_results = pd.DataFrame(results)
print(f"Total successful queries: {(df_results['status'] == '✅').sum()}")

## 5. Performance Analysis by Query Type

In [ ]:
# Group by query type
type_stats = df_results[df_results['status'] == '✅'].groupby('query_type').agg({
    'latency_ms': ['mean', 'min', 'max', 'std'],
    'memory_delta_mb': ['mean', 'min', 'max'],
    'retrieved_docs': ['mean', 'min', 'max'],
    'confidence': 'mean',
    'query_id': 'count'
}).round(3)

type_stats.columns = ['Latency_Mean', 'Latency_Min', 'Latency_Max', 'Latency_Std', 
                       'Memory_Mean', 'Memory_Min', 'Memory_Max',
                       'Docs_Mean', 'Docs_Min', 'Docs_Max',
                       'Confidence', 'Count']

print("\n📊 Performance by Query Type:")
print("="*120)
display(type_stats)

# Overall statistics
print("\n📈 Overall Statistics:")
print(f"  Average Latency: {df_results['latency_ms'].mean():.2f} ms")
print(f"  Median Latency: {df_results['latency_ms'].median():.2f} ms")
print(f"  Max Latency: {df_results['latency_ms'].max():.2f} ms")
print(f"  Average Memory Delta: {df_results['memory_delta_mb'].mean():.2f} MB")
print(f"  Average Confidence: {df_results['confidence'].mean():.3f}")

## 6. Visualization: Latency Distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Latency by query type
ax1 = axes[0, 0]
df_results[df_results['status'] == '✅'].boxplot(column='latency_ms', by='query_type', ax=ax1)
ax1.set_title('Latency Distribution by Query Type', fontsize=12, fontweight='bold')
ax1.set_xlabel('Query Type')
ax1.set_ylabel('Latency (ms)')
plt.sca(ax1)
plt.xticks(rotation=45)

# Histogram of latencies
ax2 = axes[0, 1]
ax2.hist(df_results['latency_ms'], bins=15, color='skyblue', edgecolor='black', alpha=0.7)
ax2.set_title('Latency Distribution (Histogram)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Latency (ms)')
ax2.set_ylabel('Frequency')
ax2.axvline(df_results['latency_ms'].mean(), color='red', linestyle='--', label=f'Mean: {df_results["latency_ms"].mean():.0f}ms')
ax2.legend()

# Retrieved documents
ax3 = axes[1, 0]
ax3.scatter(df_results['latency_ms'], df_results['retrieved_docs'], alpha=0.6, s=50)
ax3.set_title('Latency vs Retrieved Documents', fontsize=12, fontweight='bold')
ax3.set_xlabel('Latency (ms)')
ax3.set_ylabel('Number of Documents')
ax3.grid(True, alpha=0.3)

# Confidence vs Latency
ax4 = axes[1, 1]
scatter = ax4.scatter(df_results['latency_ms'], df_results['confidence'], 
                       c=df_results['retrieved_docs'], cmap='viridis', alpha=0.6, s=50)
ax4.set_title('Confidence vs Latency', fontsize=12, fontweight='bold')
ax4.set_xlabel('Latency (ms)')
ax4.set_ylabel('Confidence Score')
plt.colorbar(scatter, ax=ax4, label='Retrieved Docs')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('rag_latency_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Latency analysis visualization saved")

## 7. Comprehensive Query Results Table

In [ ]:
# Create comprehensive results table
display_cols = ['query_id', 'query_type', 'query', 'response', 'latency_ms', 
                 'memory_delta_mb', 'retrieved_docs', 'confidence']

df_display = df_results[display_cols].copy()
df_display['latency_ms'] = df_display['latency_ms'].round(2)
df_display['memory_delta_mb'] = df_display['memory_delta_mb'].round(3)
df_display['confidence'] = df_display['confidence'].round(3)

# Truncate for display
df_display['query'] = df_display['query'].str[:40] + '...'
df_display['response'] = df_display['response'].str[:60] + '...'

print("\n📋 Complete Query-Response Table:")
print("="*150)
display(df_display)

## 8. Query-Response Examples with Full Details

In [ ]:
# Show detailed examples for each query type
for query_type in QUERY_SETS.keys():
    df_type = df_results[df_results['query_type'] == query_type]
    
    print(f"\n{'='*100}")
    print(f"📝 {query_type.upper()} QUERIES - Examples")
    print(f"{'='*100}")
    
    for idx, row in df_type.head(2).iterrows():
        print(f"\n🔹 Query #{row['query_id']}")
        print(f"   Input: {row['query']}")
        print(f"   Output: {row['response'][:120]}...")
        print(f"   Latency: {row['latency_ms']:.2f}ms | Memory: {row['memory_delta_mb']:.2f}MB | "
              f"Docs: {row['retrieved_docs']} | Confidence: {row['confidence']:.3f}")

## 9. Memory and Performance Profiling

In [ ]:
# Profile memory usage pattern
print("\n💾 Memory Analysis:")
print("="*60)

mem_stats = df_results[df_results['status'] == '✅']['memory_delta_mb'].describe()
print(mem_stats)

# CPU profiling
proc = psutil.Process()
print(f"\n📊 Process Profiling:")
print(f"  Current Memory: {proc.memory_info().rss / (1024**2):.2f} MB")
print(f"  CPU Percent: {proc.cpu_percent(interval=1):.2f}%")
print(f"  Num Threads: {proc.num_threads()}")

# Latency percentiles
print(f"\n⏱️ Latency Percentiles:")
latency_data = df_results[df_results['status'] == '✅']['latency_ms']
for p in [50, 75, 90, 95, 99]:
    print(f"  P{p}: {np.percentile(latency_data, p):.2f}ms")

## 10. Performance Summary Report

In [ ]:
# Final summary
summary_html = f"""
<div style='background: #f0f0f0; padding: 20px; border-radius: 10px; font-family: Arial;'>
  <h2>🎯 O-RAG Performance Summary</h2>
  <table style='width: 100%; border-collapse: collapse;'>
    <tr style='background: #e0e0e0;'>
      <th style='padding: 10px; text-align: left;'>Metric</th>
      <th style='padding: 10px; text-align: left;'>Value</th>
    </tr>
    <tr>
      <td style='padding: 10px; border-bottom: 1px solid #ccc;'>Total Queries Executed</td>
      <td style='padding: 10px; border-bottom: 1px solid #ccc;'>{len(df_results)}</td>
    </tr>
    <tr>
      <td style='padding: 10px; border-bottom: 1px solid #ccc;'>Successful Queries</td>
      <td style='padding: 10px; border-bottom: 1px solid #ccc;'>{(df_results["status"] == "✅").sum()}</td>
    </tr>
    <tr>
      <td style='padding: 10px; border-bottom: 1px solid #ccc;'>Average Latency</td>
      <td style='padding: 10px; border-bottom: 1px solid #ccc;'>{df_results["latency_ms"].mean():.2f} ms</td>
    </tr>
    <tr>
      <td style='padding: 10px; border-bottom: 1px solid #ccc;'>Median Latency</td>
      <td style='padding: 10px; border-bottom: 1px solid #ccc;'>{df_results["latency_ms"].median():.2f} ms</td>
    </tr>
    <tr>
      <td style='padding: 10px; border-bottom: 1px solid #ccc;'>Max Latency</td>
      <td style='padding: 10px; border-bottom: 1px solid #ccc;'>{df_results["latency_ms"].max():.2f} ms</td>
    </tr>
    <tr>
      <td style='padding: 10px; border-bottom: 1px solid #ccc;'>Average Confidence</td>
      <td style='padding: 10px; border-bottom: 1px solid #ccc;'>{df_results["confidence"].mean():.3f}</td>
    </tr>
    <tr>
      <td style='padding: 10px; border-bottom: 1px solid #ccc;'>Avg Retrieved Docs</td>
      <td style='padding: 10px; border-bottom: 1px solid #ccc;'>{df_results["retrieved_docs"].mean():.1f}</td>
    </tr>
    <tr>
      <td style='padding: 10px;'>Test Duration</td>
      <td style='padding: 10px;'>{(df_results["timestamp"].max() - df_results["timestamp"].min()).total_seconds():.2f} seconds</td>
    </tr>
  </table>
</div>
"""

display(HTML(summary_html))